In [16]:
pip install pandas numpy geopandas folium branca requests shapely pyproj fiona rtree mapclassify --quiet

In [17]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from folium.plugins import Fullscreen, MiniMap, MeasureControl, MousePosition, Search
from branca.colormap import linear
import requests
from shapely.geometry import Point, Polygon
from pyproj import CRS, Transformer
import os
import json
import fiona
import warnings

warnings.filterwarnings('ignore') # 경고 메시지 무시

In [18]:
data_dir = 'data'
output_dir = 'output'

os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"작업 폴더 '{data_dir}' 및 '{output_dir}'가 생성되었습니다.")

작업 폴더 'data' 및 'output'가 생성되었습니다.


In [19]:
def download_data(urls, save_path, file_name, file_format=None):
    """
    주어진 URL 리스트에서 데이터를 다운로드하고 지정된 경로에 저장합니다.

    Args:
        urls (list): 데이터 다운로드를 시도할 URL 리스트 (우선순위 순).
        save_path (str): 파일을 저장할 디렉토리 경로.
        file_name (str): 저장될 파일의 이름 (확장자 포함).
        file_format (str, optional): 다운로드할 파일의 예상 형식 (예: 'geojson', 'zip').
                                     다운로드된 파일의 유효성 검사에 사용될 수 있습니다.
    Returns:
        str: 성공적으로 다운로드된 파일의 전체 경로. 다운로드 실패 시 None.
    """
    full_path = os.path.join(save_path, file_name)

    if os.path.exists(full_path):
        print(f"'{file_name}' 파일이 이미 존재하여 다운로드를 건너뜝니다.")
        return full_path

    print(f"'{file_name}' 다운로드를 시도합니다...")
    for url_idx, url in enumerate(urls):
        try:
            print(f"  - {url_idx + 1}/{len(urls)}번째 URL 시도: {url}")
            response = requests.get(url, stream=True)
            response.raise_for_status() # HTTP 오류 발생 시 예외 발생

            with open(full_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"  '{file_name}' 파일이 성공적으로 다운로드되었습니다: {full_path}")
            return full_path

        except requests.exceptions.RequestException as e:
            print(f"  URL {url} 에서 데이터를 다운로드하는 데 실패했습니다: {e}")
        except IOError as e:
            print(f"  파일을 저장하는 데 실패했습니다: {e}")
        except Exception as e:
            print(f"  예기치 않은 오류 발생: {e}")

    print(f"모든 URL에서 '{file_name}' 다운로드에 실패했습니다.")
    return None

In [20]:
pip install osmnx --quiet

In [21]:
import osmnx as ox

tags = {
    "landuse": [
        "residential",
        "commercial",
        "industrial",
        "forest"
    ]
}

gdf_landuse = ox.features_from_place(
    "Seoul, South Korea",
    tags
)

print("osmnx를 통해 서울시 토지이용 데이터 로드 성공.")
print("원본 CRS:", gdf_landuse.crs)
print(f"로드된 피처 수: {len(gdf_landuse)}")
display(gdf_landuse.head())

osmnx를 통해 서울시 토지이용 데이터 로드 성공.
원본 CRS: epsg:4326
로드된 피처 수: 4778


geometry      landuse          name  \
element id                                                                  
node    368871563    POINT (127.0145 37.48684)  residential      LG서초에클라트   
        2099195503  POINT (127.08382 37.49295)  residential       삼성사원아파트   
        2099280273  POINT (127.07148 37.48548)  residential  개포자이프레지던스아파트   
        4102269991  POINT (127.10007 37.53998)  residential      광장 현대5단지   
        5735896022  POINT (127.02286 37.50189)  residential      서초푸르지오서밋   

                   residential addr:city addr:district addr:housenumber  \
element id                                                                
node    368871563   apartments       NaN           NaN              NaN   
        2099195503  apartments     서울특별시           강남구               27   
        2099280273  apartments     서울특별시           강남구               14   
        4102269991  apartments     서울특별시           광진구              NaN   
        5735896022  apartments       NaN           NaN              NaN   

                   addr:postcode addr:street addr:subdistrict  ... architect  \
element id                                                     ...             
node    368871563            NaN         NaN              NaN  ...       NaN   
        2099195503         06339     개포로124길              일원동  ...       NaN   
        2099280273         06331         삼성로              개포동  ...       NaN   
        4102269991           NaN        아차산로              NaN  ...       NaN   
        5735896022           NaN         NaN              NaN  ...       NaN   

                   image check_date construction_date addr:city:en units  \
element id                                                                 
node    368871563    NaN        NaN               NaN          NaN   NaN   
        2099195503   NaN        NaN               NaN          NaN   NaN   
        2099280273   NaN        NaN               NaN          NaN   NaN   
        4102269991   NaN        NaN               NaN          NaN   NaN   
        5735896022   NaN        NaN               NaN          NaN   NaN   

                   alt_name_1  FID denotation type  
element id                                          
node    368871563         NaN  NaN        NaN  NaN  
        2099195503        NaN  NaN        NaN  NaN  
        2099280273        NaN  NaN        NaN  NaN  
        4102269991        NaN  NaN        NaN  NaN  
        5735896022        NaN  NaN        NaN  NaN  

[5 rows x 102 columns]

In [22]:
# 도시숲 (forest) 데이터 추출
gdf_forest = gdf_landuse[gdf_landuse['landuse'] == 'forest'].copy()

print(f"추출된 도시숲 피처 수: {len(gdf_forest)}")

# 버퍼 분석을 위해 미터 단위를 사용하는 CRS로 변환 (예: EPSG:5186 - Korea 2000 / Central Belt)
# 원본 CRS가 지리 좌표계(EPSG:4326)이므로 재투영 필요
original_crs = gdf_forest.crs
target_crs = 'EPSG:5186'

if original_crs != target_crs:
    gdf_forest_proj = gdf = gdf_forest.to_crs(target_crs)
    print(f"도시숲 데이터 CRS가 {original_crs}에서 {target_crs}로 변환되었습니다.")
else:
    gdf_forest_proj = gdf_forest
    print(f"도시숲 데이터는 이미 {target_crs} CRS입니다.")

# 500미터 버퍼 생성
# 버퍼가 겹칠 수 있으므로 dissolve(병합)하여 단일 폴리곤 또는 MultiPolygon으로 만듭니다.
# 작은 gap이 생기지 않도록 tolerance를 0으로 설정하거나, 미리 union하여 처리할 수 있습니다.
# 여기서는 단순히 버퍼링 후 dissolve합니다.

# GeoSeries에 buffer()를 적용하고, 이를 GeoDataFrame으로 변환한 후 dissolve()
gdf_forest_buffer = gdf_forest_proj.buffer(500)
gdf_forest_buffer_gdf = gpd.GeoDataFrame(geometry=gdf_forest_buffer, crs=target_crs)
gdf_forest_buffer_dissolved = gdf_forest_buffer_gdf.dissolve()

# 버퍼링된 데이터를 다시 원본 지리 좌표계(EPSG:4326)로 변환하여 Folium 시각화에 적합하게 만듭니다.
gdf_forest_buffer_4326 = gdf_forest_buffer_dissolved.to_crs(original_crs)

print("500m 도시숲 버퍼 생성 완료.")
print(f"생성된 버퍼 지오메트리 타입: {gdf_forest_buffer_4326.geometry.iloc[0].geom_type}")

display(gdf_forest_buffer_4326.head())

추출된 도시숲 피처 수: 394
도시숲 데이터 CRS가 epsg:4326에서 EPSG:5186로 변환되었습니다.
500m 도시숲 버퍼 생성 완료.
생성된 버퍼 지오메트리 타입: MultiPolygon


,geometry
0,"MULTIPOLYGON (((126.9261 37.45151, 126.92597 3..."


In [23]:
import folium
from folium.plugins import Fullscreen, MiniMap, MeasureControl, MousePosition, FloatImage
from branca.colormap import linear
import os
from jinja2 import Template

# 서울 중심 좌표 설정 (위도, 경도)
seoul_center = [37.5665, 126.9780]

# Folium 지도 초기화
m = folium.Map(location=seoul_center, zoom_start=11, tiles='CartoDB positron')

# 전체 토지이용 데이터 (gdf_landuse) 추가
def style_function(feature):
    landuse_type = feature['properties'].get('landuse', 'unknown')
    color_map = {
        'residential': '#e6f598',
        'commercial': '#fee08b',
        'industrial': '#fc8d59',
        'forest': '#66bd63',
        'unknown': '#d73027'
    }
    return {
        'fillColor': color_map.get(landuse_type, '#d73027'),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.7
    }

folium.GeoJson(
    gdf_landuse.to_json(),
    name='서울시 토지이용 현황 (OSM)',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['landuse', 'name'], aliases=['토지이용', '이름']),
).add_to(m)

# 도시숲 500m 버퍼 데이터 (gdf_forest_buffer_4326) 추가
folium.GeoJson(
    gdf_forest_buffer_4326.to_json(),
    name='도시숲 500m 버퍼',
    style_function=lambda x: {
        'fillColor': '#9ecae1',
        'color': 'blue',
        'weight': 1,
        'fillOpacity': 0.5
    }
).add_to(m)

# 레이어 컨트롤 추가
folium.LayerControl().add_to(m)

# 미니맵 추가
MiniMap().add_to(m)

# 풀스크린 컨트롤 추가
Fullscreen().add_to(m)

# 거리 측정 도구 추가
MeasureControl(position='topright').add_to(m)

# 마우스 위치 좌표 표시
MousePosition(position='bottomright', separator=' | ', num_decimals=4,
              empty_string='LatLng', prefix='Coordinates: ').add_to(m)

In [24]:
from branca.element import Template, MacroElement

# =========================
# 범례
# =========================

legend_template = """
{% macro html(this, kwargs) %}

<div style="
position: fixed;
bottom: 15px;
left: 15px;
z-index:9999;

background:white;

padding:8px;

border:1px solid gray;
border-radius:6px;

font-size:11px;

box-shadow:0 0 8px rgba(0,0,0,0.2);
">

<div style="
font-weight:bold;
font-size:12px;
margin-bottom:5px;
">
범례
</div>

<div>
<span style="
background:#e6f598;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
주거지역
</div>

<div>
<span style="
background:#fee08b;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
상업지역
</div>

<div>
<span style="
background:#fc8d59;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
산업지역
</div>

<div>
<span style="
background:#66bd63;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
도시숲
</div>

<div>
<span style="
background:#d73027;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
미지정
</div>

<div>
<span style="
background:#9ecae1;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
border:1px solid blue;
"></span>
500m 버퍼
</div>

</div>

{% endmacro %}
"""

legend = MacroElement()
legend._template = Template(legend_template)

m.get_root().add_child(legend)


output_map_path_with_legend = os.path.join(output_dir, 'seoul_landuse_forest_map_with_legend.html')
m.save(output_map_path_with_legend)

print(f"인터랙티브 지도가 '{output_map_path_with_legend}' 파일로 저장되었습니다.")

인터랙티브 지도가 'output/seoul_landuse_forest_map_with_legend.html' 파일로 저장되었습니다.


In [15]:
import geopandas as gpd
import folium
from folium.plugins import Fullscreen, MiniMap, MeasureControl, MousePosition
from branca.element import Template, MacroElement
import os

# --- 1. 상업지역 및 산업지역 데이터 추출 및 버퍼 생성 ---
# 기존 gdf_landuse에서 상업지역과 산업지역 데이터 추출
gdf_commercial = gdf_landuse[gdf_landuse['landuse'] == 'commercial'].copy()
gdf_industrial = gdf_landuse[gdf_landuse['landuse'] == 'industrial'].copy()

# 버퍼링을 위한 CRS 및 거리 설정 (이전 셀에서 사용된 값 재활용)
original_crs = gdf_landuse.crs # EPSG:4326
target_crs = 'EPSG:5186'
buffer_distance = 500 # meters

def create_and_reproject_buffer(gdf, landuse_type, buffer_distance, original_crs, target_crs):
    print(f"\n{landuse_type} 데이터 처리 시작...")
    print(f"추출된 {landuse_type} 피처 수: {len(gdf)}")

    if gdf.empty:
        print(f"경고: {landuse_type} 데이터가 비어 있습니다. 버퍼를 생성하지 않습니다.")
        return gpd.GeoDataFrame(geometry=[], crs=original_crs) # 비어있는 GeoDataFrame 반환

    # 버퍼 생성을 위해 투영 CRS로 변환
    if original_crs != target_crs:
        gdf_proj = gdf.to_crs(target_crs)
        print(f"{landuse_type} 데이터 CRS가 {original_crs}에서 {target_crs}로 변환되었습니다.")
    else:
        gdf_proj = gdf
        print(f"{landuse_type} 데이터는 이미 {target_crs} CRS입니다.")

    # 버퍼 생성 후 dissolve (병합)
    gdf_buffer = gdf_proj.buffer(buffer_distance)
    gdf_buffer_gdf = gpd.GeoDataFrame(geometry=gdf_buffer, crs=target_crs)
    gdf_buffer_dissolved = gdf_buffer_gdf.dissolve()

    # Folium 시각화를 위해 다시 원본 지리 좌표계(EPSG:4326)로 변환
    gdf_buffer_4326 = gdf_buffer_dissolved.to_crs(original_crs)
    print(f"{buffer_distance}m {landuse_type} 버퍼 생성 완료.")
    print(f"생성된 버퍼 지오메트리 타입: {gdf_buffer_4326.geometry.iloc[0].geom_type if not gdf_buffer_4326.empty else 'Empty'}")
    return gdf_buffer_4326

gdf_commercial_buffer_4326 = create_and_reproject_buffer(gdf_commercial, "상업지역", buffer_distance, original_crs, target_crs)
gdf_industrial_buffer_4326 = create_and_reproject_buffer(gdf_industrial, "산업지역", buffer_distance, original_crs, target_crs)

# --- 2. Folium 지도 재생성 및 모든 레이어, 컨트롤, 범례 추가 ---

# 서울 중심 좌표 설정 (이전 셀에서 재활용)
seoul_center = [37.5665, 126.9780]

# Folium 지도 초기화 (새로 생성하여 깨끗한 상태로 시작)
m = folium.Map(location=seoul_center, zoom_start=11, tiles='CartoDB positron')

# 전체 토지이용 데이터 (gdf_landuse) 추가 (이전 셀에서 재활용)
def style_function(feature):
    landuse_type = feature['properties'].get('landuse', 'unknown')
    color_map = {
        'residential': '#e6f598',
        'commercial': '#fee08b',
        'industrial': '#fc8d59',
        'forest': '#66bd63',
        'unknown': '#d73027'
    }
    return {
        'fillColor': color_map.get(landuse_type, '#d73027'),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.7
    }

folium.GeoJson(
    gdf_landuse.to_json(),
    name='서울시 토지이용 현황 (OSM)',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['landuse', 'name'], aliases=['토지이용', '이름']),
).add_to(m)

# 도시숲 500m 버퍼 데이터 (gdf_forest_buffer_4326) 추가 (이전 셀에서 재활용)
if 'gdf_forest_buffer_4326' in locals() and not gdf_forest_buffer_4326.empty:
    folium.GeoJson(
        gdf_forest_buffer_4326.to_json(),
        name='도시숲 500m 버퍼',
        style_function=lambda x: {
            'fillColor': '#9ecae1',
            'color': 'blue',
            'weight': 1,
            'fillOpacity': 0.5
        }
    ).add_to(m)
    print("도시숲 500m 버퍼 레이어가 지도에 추가되었습니다.")
else:
    print("도시숲 버퍼 데이터가 없거나 비어있어 레이어를 추가하지 않았습니다.")

# 상업지역 500m 버퍼 데이터 추가 (신규)
if not gdf_commercial_buffer_4326.empty:
    folium.GeoJson(
        gdf_commercial_buffer_4326.to_json(),
        name='상업지역 500m 버퍼',
        style_function=lambda x: {
            'fillColor': '#a6d854', # 상업지역 버퍼를 위한 고유 색상
            'color': '#7fc97f',
            'weight': 1,
            'fillOpacity': 0.5
        }
    ).add_to(m)
    print("상업지역 500m 버퍼 레이어가 지도에 추가되었습니다.")
else:
    print("상업지역 버퍼 데이터가 없어 레이어를 추가하지 않았습니다.")

# 산업지역 500m 버퍼 데이터 추가 (신규)
if not gdf_industrial_buffer_4326.empty:
    folium.GeoJson(
        gdf_industrial_buffer_4326.to_json(),
        name='산업지역 500m 버퍼',
        style_function=lambda x: {
            'fillColor': '#e78ac3', # 산업지역 버퍼를 위한 고유 색상
            'color': '#ff7f00',
            'weight': 1,
            'fillOpacity': 0.5
        }
    ).add_to(m)
    print("산업지역 500m 버퍼 레이어가 지도에 추가되었습니다.")
else:
    print("산업지역 버퍼 데이터가 없어 레이어를 추가하지 않았습니다.")

# 레이어 컨트롤 및 기타 기능 추가 (이전 셀에서 재활용)
folium.LayerControl().add_to(m)
MiniMap().add_to(m)
Fullscreen().add_to(m)
MeasureControl(position='topright').add_to(m)
MousePosition(position='bottomright', separator=' | ', num_decimals=4,
              empty_string='LatLng', prefix='Coordinates: ').add_to(m)

# --- 3. 범례 업데이트 및 지도에 추가 ---
legend_template_final = """
{% macro html(this, kwargs) %}

<div style="
position: fixed;
bottom: 15px;
left: 15px;
z-index:9999;

background:white;

padding:8px;

border:1px solid gray;
border-radius:6px;

font-size:11px;

box-shadow:0 0 8px rgba(0,0,0,0.2);
">

<div style="
font-weight:bold;
font-size:12px;
margin-bottom:5px;
">
범례
</div>

<div>
<span style="
background:#e6f598;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
주거지역
</div>

<div>
<span style="
background:#fee08b;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
상업지역
</div>

<div>
<span style="
background:#fc8d59;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
산업지역
</div>

<div>
<span style="
background:#66bd63;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
도시숲
</div>

<div>
<span style="
background:#d73027;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
"></span>
미지정
</div>

<div>
<span style="
background:#9ecae1;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
border:1px solid blue;
"></span>
도시숲 500m 버퍼
</div>

<div>
<span style="
background:#a6d854;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
border:1px solid #7fc97f;
"></span>
상업지역 500m 버퍼
</div>

<div>
<span style="
background:#e78ac3;
width:10px;
height:10px;
display:inline-block;
margin-right:4px;
border:1px solid #ff7f00;
"></span>
산업지역 500m 버퍼
</div>

</div>

{% endmacro %}
"""

legend_final = MacroElement()
legend_final._template = Template(legend_template_final)
m.get_root().add_child(legend_final)

# --- 4. 최종 지도 저장 및 표시 ---
output_map_path_with_legend = os.path.join(output_dir, 'seoul_landuse_forest_map_with_commercial_industrial_buffers.html')
m.save(output_map_path_with_legend)

print(f"인터랙티브 지도가 '{output_map_path_with_legend}' 파일로 저장되었습니다. 이제 상업지역 및 산업지역 버퍼가 포함됩니다.")


상업지역 데이터 처리 시작...
추출된 상업지역 피처 수: 417
상업지역 데이터 CRS가 epsg:4326에서 EPSG:5186로 변환되었습니다.
500m 상업지역 버퍼 생성 완료.
생성된 버퍼 지오메트리 타입: MultiPolygon

산업지역 데이터 처리 시작...
추출된 산업지역 피처 수: 172
산업지역 데이터 CRS가 epsg:4326에서 EPSG:5186로 변환되었습니다.
500m 산업지역 버퍼 생성 완료.
생성된 버퍼 지오메트리 타입: MultiPolygon
도시숲 500m 버퍼 레이어가 지도에 추가되었습니다.
상업지역 500m 버퍼 레이어가 지도에 추가되었습니다.
산업지역 500m 버퍼 레이어가 지도에 추가되었습니다.
인터랙티브 지도가 'output/seoul_landuse_forest_map_with_commercial_industrial_buffers.html' 파일로 저장되었습니다. 이제 상업지역 및 산업지역 버퍼가 포함됩니다.


In [25]:
from google.colab import files

# output_map_path_with_legend 변수에 저장된 경로를 사용하여 파일 다운로드
files.download(output_map_path_with_legend)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>